# NoFluffJobs API search

Query the NoFluffJobs posting search endpoint, inspect the response, and collect the returned postings into a clean list.

## 1. Imports

We use `httpx` for the HTTP request and `pandas` later for tabular display.

In [4]:
import httpx
import pandas as pd

## 2. Endpoint and query parameters

`params` are sent as query string arguments. Salary is requested in EUR per month. The `region`/`requirement` filters are left commented so the search stays broad.

In [5]:
url = "https://nofluffjobs.com/api/search/posting"

# Query parameters sent on the URL
params = {
    "salaryCurrency": "EUR",
    "salaryPeriod": "month",
    "region": "nl",          # 'nl', 'pl', 'cz', 'hu', etc.
    "language": "en",
    "page": 1,
}

## 3. Request body

The backend expects the key `criteriaSearch` (not `criteria`). Add filters such as `requirement` inside it to narrow the search.

In [6]:
payload = {
    "criteriaSearch": {
        # "requirement": ["python"]
    }
}

## 4. HTTP headers

A browser-like `User-Agent` plus `Referer`/`Origin` are required for the API to accept the request.

In [7]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Referer": "https://nofluffjobs.com/",
    "Origin": "https://nofluffjobs.com",
}

## 5. Send the request

In [8]:
# Query the first 10 pages
all_postings = []
for page in range(1, 40):
    params["page"] = page
    response = httpx.post(url, params=params, json=payload, headers=headers)
    print(f"page {page}: HTTP {response.status_code}")
    if response.status_code == 200:
        data = response.json()
        postings = data.get("postings", [])
        all_postings.extend(postings)
        print(f"  retrieved {len(postings)} postings")
    else:
        print(f"  failed: {response.text[:200]}")
print(f"Total postings collected: {len(all_postings)}")

page 1: HTTP 200
  retrieved 136 postings
page 2: HTTP 200
  retrieved 197 postings
page 3: HTTP 200
  retrieved 211 postings
page 4: HTTP 200
  retrieved 197 postings
page 5: HTTP 200
  retrieved 164 postings
page 6: HTTP 200
  retrieved 123 postings
page 7: HTTP 200
  retrieved 120 postings
page 8: HTTP 200
  retrieved 104 postings
page 9: HTTP 200
  retrieved 209 postings
page 10: HTTP 200
  retrieved 139 postings
page 11: HTTP 200
  retrieved 216 postings
page 12: HTTP 200
  retrieved 176 postings
page 13: HTTP 200
  retrieved 123 postings
page 14: HTTP 200
  retrieved 138 postings
page 15: HTTP 200
  retrieved 112 postings
page 16: HTTP 200
  retrieved 151 postings
page 17: HTTP 200
  retrieved 77 postings
page 18: HTTP 200
  retrieved 124 postings
page 19: HTTP 200
  retrieved 123 postings
page 20: HTTP 200
  retrieved 205 postings
page 21: HTTP 200
  retrieved 187 postings
page 22: HTTP 200
  retrieved 220 postings
page 23: HTTP 200
  retrieved 99 postings
page 24: HTTP 200
  re

## 6. Parse and clean the response

On success, extract the `postings` list. Each posting is kept as-is for now; `salary` is read out to confirm it is present.

In [9]:
# Keep all retrieved postings
cleaned_postings = all_postings

## 7. Inspect the collected postings

In [10]:
# display(cleaned_postings)

### Optional: view as a DataFrame

Flatten a few key fields for easier inspection.

In [11]:
if cleaned_postings:
    df = pd.DataFrame([
        {
            "title": j.get("title"),
            "company": j.get("name"),
            "salary.from": j.get("salary", {}).get("from"),
            "salary.to": j.get("salary", {}).get("to"),
            "currency": j.get("salary", {}).get("currency"),
            "remote": j.get("fullyRemote"),
        }
        for j in cleaned_postings
    ])
    display(df)
else:
    print("No postings collected.")

,title,company,salary.from,salary.to,currency,remote
0,Sr. Software Engineer,Bamboo Health Sp. z o.o.,3370.0,3969.0,EUR,False
1,Sr. Software Engineer,Bamboo Health Sp. z o.o.,3370.0,3969.0,EUR,False
2,Sr. Software Engineer,Bamboo Health Sp. z o.o.,3370.0,3969.0,EUR,False
3,Sr. Software Engineer,Bamboo Health Sp. z o.o.,3370.0,3969.0,EUR,False
4,Sr. Software Engineer,Bamboo Health Sp. z o.o.,3370.0,3969.0,EUR,False
...,...,...,...,...,...,...
5976,React Native Developer,Connectis_,5376.0,6216.0,EUR,False
5977,React Native Developer,Connectis_,5376.0,6216.0,EUR,False
5978,Senior Fullstack Developer (Java + Angular),"AVENGA (Agencja Pracy, nr KRAZ: 8448)",4876.0,6464.0,EUR,False
5979,Senior Engineer (Cloud / D-Fend),Motorola Solutions Systems Polska,3175.0,4536.0,EUR,False


## 8. Persist to SQLite

Store each posting in a local SQLite database.

In [12]:
import sqlite3
import json as _json
import os

db_path = "job_postings.db"
# wipe old database so schema changes take effect
if os.path.exists(db_path):
    os.remove(db_path)
conn = sqlite3.connect(db_path)

# Flatten every field of each posting into columns, keeping all of them
df_all = pd.json_normalize(cleaned_postings)

# Make nested values (lists/dicts) storable in SQLite
df_all = df_all.map(lambda x: _json.dumps(x) if isinstance(x, (list, dict)) else x)

# Use a composite key to deduplicate regional variants of the same posting
df_all["pk"] = df_all["name"].astype(str) + "|" + df_all["title"].astype(str) + "|" + df_all["category"].astype(str)
df_all = df_all.set_index("pk")

# Create the table with the composite primary key
cols = ["pk"] + [c for c in df_all.columns]
col_defs = ", ".join(["pk TEXT PRIMARY KEY"] + [f'"{c}"' for c in df_all.columns])
conn.execute(f"CREATE TABLE nofluff ({col_defs})")

# Upsert: insert new rows, keep older records on conflict
placeholders = ", ".join(["?"] * len(cols))
col_names = ", ".join([f'"{c}"' for c in cols])
sql = f"INSERT OR IGNORE INTO nofluff ({col_names}) VALUES ({placeholders})"
for idx, row in df_all.iterrows():
    values = [idx] + [row[c] for c in df_all.columns]
    conn.execute(sql, values)
conn.commit()

cur = conn.cursor()
print(f"Saved postings; total in nofluff: {cur.execute('SELECT COUNT(*) FROM nofluff').fetchone()[0]}")
conn.close()


Saved postings; total in nofluff: 758
